# 第4周 · 周二：数仓建模（一）——四层分层与维度建模

> 难度：⭐⭐⭐⭐ | 定位：理解数仓四层分层的设计原理，掌握星型模型和维度建模方法论
> 前置依赖：周一的"数据流转"认知 | 后续用途：数仓表结构设计、与数仓工程师高效协作
> 面试地位：大厂数据分析面试必考，约占数据工程类问题的30%

---

## 📘 区块1：核心概念与第一性原理

### 1.1 数仓为什么要分层——从第一性原理推导

假设你是公司第一个数据分析师。公司只有一个MySQL数据库，所有数据都在里面。你每天写SQL做分析，一切正常。

三个月后，公司发展到100人。你发现：
- 数据太脏：订单表里的手机号有的带+86有的不带，有的用半角有的用全角
- 计算太慢：每次做月度分析都要从头清洗一遍数据
- 口径混乱：你算的GMV是下单金额，财务算的GMV是实付金额，运营算的GMV是下单-退款
- 依赖混乱：你不知道哪个表是"源头"，哪个表是"中间结果"，改了一个字段不知道会影响到哪些报表

这些问题都指向同一个根因——**缺少分层**。

分层的本质是什么？**把"清洗"和"计算"的工作在数据入库时就做好，让每个数据使用者拿到手的就是"干净且口径统一"的数据。**

更准确地说，分层解决了四个核心问题：
1. **清晰度问题**：每层有明确的职责，谁都知道数据在哪
2. **复用性问题**：清洗一次，所有人用，不用每人重新清洗
3. **一致性问题**：同一指标只有一个口径，所有人都从同一层取
4. **效率问题**：预聚合的数据让查询从分钟级降到秒级

### 1.2 四层分层的设计哲学

数仓的经典四层结构是数十年工业实践的结晶，每一层都有其独特的存在理由：

```
ODS（Operational Data Store —— 操作数据层）
  ↓  职责：照搬原始数据，不做任何修改
  ↓  特点：最"脏"但最"完整"
  ↓  命名：ods_{源系统}_{源表名}_{增量/全量标识}
  ↓
DWD（Data Warehouse Detail —— 明细数据层）
  ↓  职责：清洗 + 标准化，保留明细粒度
  ↓  特点：干净、标准、分析师最常用的明细层
  ↓  命名：dwd_{主题域}_{业务过程}_{增量/全量标识}
  ↓
DWS（Data Warehouse Summary —— 汇总数据层）
  ↓  职责：轻度汇总，面向主题聚合
  ↓  特点：快、但丢失明细
  ↓  命名：dws_{主题域}_{维度}_{时间粒度}
  ↓
ADS（Application Data Service —— 应用数据层）
  ↓  职责：面向具体应用场景的定制化数据
  ↓  特点：极快、定制化、直接喂给看板
  ↓  命名：ads_{应用场景}_{报表名}
```

**关键设计原则**：
- **单向依赖**：上层可以依赖下层，下层绝不能依赖上层
- **同层独立**：同一层的表之间应该尽量独立，避免循环依赖
- **渐进清洗**：ODS最脏→DWD干净→DWS聚合→ADS定制，每一步都在增加"价值"

### 1.3 为什么ODS不能跳过

有人问："既然ODS就是照搬数据，为什么不直接把业务库数据清洗好再搬过来？"

原因有三：
1. **可追溯**：如果清洗逻辑有bug，有ODS可以随时用原始数据重新清洗，不用去找业务库要数据
2. **解耦业务库**：ODS一旦同步完成，后续的所有处理都不再访问业务库，保护业务库不受分析负载影响
3. **断点恢复**：ETL任务失败了？直接从ODS重新启动，不影响业务库

ODS的"不做任何修改"恰恰是它最大的价值——它是数仓的"定海神针"，是数据可追溯性的基石。

### 1.4 事实表 vs 维度表——两种本质不同的表

数仓中最重要的分类就是"事实表"和"维度表"：

**事实表（Fact Table）**：
- 记录的是"发生了什么"——每一行是一个业务事件
- 特征：行数极多（亿到万亿）、列相对少、不断增长
- 经典例子：订单表、支付表、登录表、浏览日志
- 关键字段：度量值（金额、数量、时长）+ 维度外键（user_id, product_id, date_id）

**维度表（Dimension Table）**：
- 记录的是"谁、什么、何时、何地"——描述业务实体的属性
- 特征：行数相对少（万到百万）、列多（描述丰富）、变化缓慢
- 经典例子：用户表、商品表、店铺表、日期表、地区表
- 关键字段：主键 + 描述性属性（名称、类别、层级、状态）

| 对比维度 | 事实表 | 维度表 |
|---------|--------|--------|
| 记录内容 | 业务事件 | 实体属性 |
| 数据量 | 海量（亿-万亿行） | 少量（万-百万行） |
| 增长速度 | 快速增长 | 缓慢增长 |
| 列数量 | 少（10-30列） | 多（50-200列） |
| 更新模式 | 追加（INSERT） | 缓慢变化（UPDATE/INSERT） |
| 典型操作 | 聚合、求和、计数 | 过滤、分组、关联 |

### 1.5 面试高频考点

**面试官常问**：
- "数仓为什么分四层？每层的作用是什么？"
- "ODS层能跳过吗？为什么？"
- "事实表和维度表有什么区别？"
- "星型模型和雪花模型的区别和适用场景？"
- "你怎么决定一个指标该放在DWD还是DWS？"

**回答要点**：
- 分层解决清晰度、复用性、一致性、效率四大问题
- ODS不能跳过——它是可追溯性和断点恢复的基础
- 事实表记录事件（INSERT为主），维度表记录实体属性（UPDATE为主）
- 星型模型简单高效（维度表冗余但查询快），雪花模型规范（节省空间但JOIN多）
- DWD保留明细（灵活但慢），DWS预聚合（快但不灵活），高频使用的汇总应放DWS

## 🧩 区块2：知识点详细拆解

### 2.1 ODS层详解

**为什么重要**：ODS是所有下游数据的唯一源头，它的设计直接决定了数仓的数据完整性。

**ODS层的设计原则**：
1. **不做业务逻辑转换**：数据从业务库来什么样就存什么样
2. **保留所有数据**：即使你认为某些字段"没用"，也不要删
3. **添加技术元数据**：增加`etl_time`（数据同步时间）、`ds`（分区日期）等技术字段
4. **保持增量/全量策略一致**：每张表明确是增量同步还是全量快照

**常见ODS表结构示例**：
```sql
-- ods_order_log_df（订单日志全量表）
CREATE TABLE ods_order_log_df (
    id BIGINT COMMENT '自增ID（来自源表）',
    order_id BIGINT COMMENT '订单ID',
    user_id BIGINT COMMENT '用户ID',
    amount DECIMAL(10,2) COMMENT '订单金额',
    status VARCHAR(20) COMMENT '订单状态',
    create_time DATETIME COMMENT '创建时间',
    update_time DATETIME COMMENT '更新时间',
    -- 以下为ETL添加的技术字段
    etl_time DATETIME COMMENT 'ETL同步时间',
    ds VARCHAR(10) COMMENT '分区日期'
) PARTITIONED BY (ds);
```

### 2.2 DWD层详解

**为什么重要**：DWD是分析师最常打交道的层，它的质量决定了所有分析的质量。

**DWD层的核心工作**：

**1. 字段标准化**
- `user_id` vs `uid` → 统一为`user_id`
- `phone` vs `mobile` vs `tel` → 统一为`phone`
- 枚举值：`'男'/'女'` vs `'M'/'F'` vs `1/2` → 统一为一种

**2. 空值处理**
- 金额字段NULL → 0（表示没有金额）还是保留NULL（表示未知）？
- 字符串字段NULL → ''（空串）还是保留NULL？
- **原则**：NULL表示"未知"，0表示"零值"——不该混淆

**3. 脏数据过滤**
- 订单金额为负 → 可能是退款，需要标记而不是删除
- 未来日期（`order_date > CURRENT_DATE()`）→ 可能是测试数据
- 明显异常的数值（如age=9999）

**4. 口径统一**
- GMV到底包含什么？（下单金额？实付金额？含运费？含退款？）
- 活跃用户的定义？（打开App？有行为？有支付？）
- **关键**：在DWD层就把这些口径定死，下游一律按此口径

### 2.3 DWS层详解

**为什么重要**：DWS是"快"的保证——预聚合的数据让日常分析从分钟级降到秒级。

**DWS层的汇总策略**：

| 汇总维度 | 表名示例 | 粒度 |
|---------|---------|------|
| 时间 | dws_category_daily_1d | 品类 + 天 |
| 用户 | dws_user_daily_1d | 用户 + 天 |
| 地域 | dws_city_daily_1d | 城市 + 天 |
| 渠道 | dws_channel_daily_1d | 渠道 + 天 |
| 多维 | dws_category_city_daily_1d | 品类 + 城市 + 天 |

**汇总的核心矛盾**：
- 维度越多 → 表越多 → 存储和ETL成本越大
- 维度越少 → 分析时可能需要回退到DWD层 → 查询慢
- 解决：只汇总"最常被查询的维度组合"

**什么该进DWS的判断标准**：
1. 查询频率高（每天>10次）
2. 查询延迟要求高（必须<1秒）
3. 维度相对固定（不需要频繁变更）
4. 数据量大（不预聚合的话查询很慢）

### 2.4 ADS层详解

**为什么重要**：ADS是"最后一公里"——数据从"能用"变成"好用"的关键。

**ADS层的典型形态**：

| 类型 | 示例 | 特点 |
|------|------|------|
| 看板数据表 | ads_ceo_dashboard_daily | 每天一行，所有CEO关心的指标 |
| 报表数据表 | ads_operation_weekly | 每周一行，运营周报用 |
| 宽表 | ads_user_wide | 把用户的所有信息拼成一行 |
| 接口数据表 | ads_api_recommend | 直接提供给推荐算法调用 |

**ADS层的设计原则**：
- **反范式化**：为了查询速度可以大量冗余
- **定制化**：每个应用场景一张表，不追求通用
- **简洁化**：字段名应该业务人员也能看懂，而不是技术缩写

### 2.5 星型模型详解

**为什么重要**：星型模型是数仓最经典的建模方式，理解它才能理解"为什么表要这样设计"。

**星型模型的结构**：
```
              ┌─────────────┐
              │  用户维度表  │
              │  user_id    │
              │  name       │
              │  city       │
              │  level      │
              └──────┬──────┘
                     │
    ┌─────────────┐  │  ┌─────────────┐
    │  时间维度表  │  │  │  商品维度表  │
    │  date_id    │  │  │  product_id │
    │  year       │  │  │  name       │
    │  month      │  │  │  category   │
    │  day        │  │  │  price      │
    └──────┬──────┘  │  └──────┬──────┘
           │         │         │
           ▼         ▼         ▼
    ┌─────────────────────────────────┐
    │          订单事实表              │
    │  order_id (PK)                 │
    │  user_id (FK → 用户维度)       │
    │  product_id (FK → 商品维度)    │
    │  date_id (FK → 时间维度)       │
    │  amount (度量)                 │
    │  quantity (度量)               │
    └─────────────────────────────────┘
```

**星型模型的优点**：
- 结构简单直观：事实表在中间，维度表围着
- 查询高效：通常只需要事实表JOIN一层维度表
- 易于理解：业务人员也能看懂这个结构

**星型模型的缺点**：
- 维度表冗余大：维度表不做规范化，存在冗余
- 不适合频繁变化的维度：维度变化时历史数据怎么处理？

### 2.6 雪花模型 vs 星型模型

**雪花模型**是星型模型的规范化版本——维度表进一步拆分成子维度表：

```
星型：商品维度表直接包含 category_name
雪花：商品维度表只包含 category_id → 需要再JOIN category维度表
```

| 对比维度 | 星型模型 | 雪花模型 |
|---------|---------|----------|
| 结构 | 一层维度 | 多层维度（像雪花分叉） |
| 冗余 | 高（维度表冗余） | 低（规范化） |
| JOIN数量 | 少（通常1层） | 多（可能2-3层） |
| 查询性能 | 好 | 差（多一层JOIN） |
| 存储成本 | 高 | 低 |
| 维护复杂度 | 低 | 高 |
| 适用场景 | 分析型数仓（主流） | 规范化要求高的场景 |

**业界实践**：95%的分析型数仓采用星型模型。存储成本远低于查询性能的价值——多存几个TB的数据比每次查询多等10秒划算得多。

### 2.7 宽表 vs 窄表的设计权衡

分析师最喜欢"宽表"——一张表里什么字段都有，不用JOIN。但宽表不是免费的午餐：

| 维度 | 窄表（DWD风格） | 宽表（ADS风格） |
|------|---------------|---------------|
| 列数 | 少（10-30列） | 多（100-500列） |
| 行数 | 多（事件粒度） | 少（聚合后） |
| 更新效率 | 高 | 低（冗余太多，更新一个维度要更新全表） |
| 查询效率 | 需要JOIN | 直接SELECT |
| 存储空间 | 省 | 费（大量重复） |
| 灵活性 | 高（维度独立变更） | 低（结构定了难改） |
| 适用场景 | 底层数仓 | 上层应用 |

**设计原则**：
- DWD层用窄表（保证规范性和灵活性）
- ADS层按需做宽表（方便具体应用直接使用）
- DWS层在两者之间（轻度聚合，适度拓宽）

### 2.8 数据分层的常见反模式

| 反模式 | 表现 | 后果 |
|--------|------|------|
| 层层拷贝 | ODS→DWD→DWS只是SELECT *，没有清洗和聚合 | 分层沦为空壳，查询照样慢 |
| 跨层调用 | ADS直接从ODS取数据 | 分层形同虚设，口径混乱 |
| 循环依赖 | A层表依赖B层表，B层又依赖A层 | ETL死锁，数据永远跑不出来 |
| 过度分层 | 分了7-8层 | 数据链路太长，维护成本爆炸 |
| 跳过分层 | 所有数据都在一层 | 回到起点：数据脏、查询慢、口径乱 |

## 💻 区块3：动手练习

### 步骤0：准备——识别你身边的"分层"

In [1]:
# ===== 步骤0：盘点你日常使用的表属于哪一层 =====

print("=" * 60)
print("任务：给你的常用表做'分层体检'")
print("=" * 60)
print()

# 示例判断逻辑
def classify_table(table_name):
    """根据表名前缀判断数仓层级"""
    if table_name.startswith("ods_"):
        return "ODS（操作数据层）—— 原始数据，不要直接用"
    elif table_name.startswith("dwd_"):
        return "DWD（明细数据层）—— 干净明细，分析师最常用"
    elif table_name.startswith("dws_"):
        return "DWS（汇总数据层）—— 轻度聚合，日常分析首选"
    elif table_name.startswith("ads_"):
        return "ADS（应用数据层）—— 定制化数据，直接喂看板"
    elif table_name.startswith("dim_"):
        return "DIM（维度表）—— 实体描述，配合事实表使用"
    elif table_name.startswith("tmp_"):
        return "TMP（临时表）—— 中间结果，不要依赖！"
    else:
        return "未知层级 —— 建议问数仓同事或查数据字典"

# 模拟一个分析师常用的表列表
tables_you_might_use = [
    "dwd_order_detail_df",
    "dws_user_daily_1d",
    "dim_user",
    "ods_order_log_df",
    "ads_ceo_dashboard_daily",
    "tmp_20260115_user_temp",
    "dwd_user_register_df",
    "dws_category_daily_1d",
]

print("常见数仓表的层级判断：")
print()
for t in tables_you_might_use:
    layer = classify_table(t)
    print(f"  {t:<35} → {layer}")

print()
print("注意：")
print("  ✓ 日常分析优先用DWS层（快）")
print("  ✓ 需要明细数据时才用DWD层（灵活但慢）")
print("  ✗ 不要直接用ODS层（脏、没清洗）")
print("  ✗ 不要依赖tmp_表（随时可能被删）")

任务：给你的常用表做'分层体检'

常见数仓表的层级判断：

  dwd_order_detail_df                 → DWD（明细数据层）—— 干净明细，分析师最常用
  dws_user_daily_1d                   → DWS（汇总数据层）—— 轻度聚合，日常分析首选
  dim_user                            → DIM（维度表）—— 实体描述，配合事实表使用
  ods_order_log_df                    → ODS（操作数据层）—— 原始数据，不要直接用
  ads_ceo_dashboard_daily             → ADS（应用数据层）—— 定制化数据，直接喂看板
  tmp_20260115_user_temp              → TMP（临时表）—— 中间结果，不要依赖！
  dwd_user_register_df                → DWD（明细数据层）—— 干净明细，分析师最常用
  dws_category_daily_1d               → DWS（汇总数据层）—— 轻度聚合，日常分析首选

注意：
  ✓ 日常分析优先用DWS层（快）
  ✓ 需要明细数据时才用DWD层（灵活但慢）
  ✗ 不要直接用ODS层（脏、没清洗）
  ✗ 不要依赖tmp_表（随时可能被删）


### 步骤1：基础练习——设计星型模型

In [2]:
# ===== 步骤1：为电商平台设计星型模型 =====

print("=" * 60)
print("任务：为电商平台设计订单分析的星型模型")
print("=" * 60)
print()

print("【事实表】dwd_order_detail —— 订单明细事实表")
print("-" * 40)
print("度量字段（可以求和/平均的数值）：")
print("  1. amount         DECIMAL —— 订单金额")
print("  2. quantity        INT     —— 购买数量")
print("  3. discount_amount DECIMAL —— 优惠金额")
print("  4. shipping_fee    DECIMAL —— 运费")
print("  5. pay_amount      DECIMAL —— 实付金额")
print()
print("维度外键（关联到维度表的ID）：")
print("  1. user_id         BIGINT  → dim_user")
print("  2. product_id      BIGINT  → dim_product")
print("  3. shop_id         BIGINT  → dim_shop")
print("  4. order_date      DATE    → dim_date")
print("  5. city_id         INT     → dim_city")
print("  6. channel_id      INT     → dim_channel")
print()

print("【维度表】")
print("-" * 40)
print("dim_user（用户维度）：")
print("  user_id, name, phone, city, level, register_date, ...")
print()
print("dim_product（商品维度）：")
print("  product_id, name, category, brand, price, status, ...")
print()
print("dim_shop（店铺维度）：")
print("  shop_id, name, owner, city, open_date, level, ...")
print()
print("dim_date（时间维度）：")
print("  date_id, year, month, day, weekday, is_holiday, week_of_year, quarter, ...")
print()

print("【查询示例】各品类 × 城市的GMV")
print("-" * 40)
print("""
SELECT
    p.category,
    c.city_name,
    SUM(o.amount) AS gmv,
    COUNT(DISTINCT o.user_id) AS buyers
FROM dwd_order_detail o
JOIN dim_product p ON o.product_id = p.product_id
JOIN dim_city c ON o.city_id = c.city_id
WHERE o.order_date >= '2026-01-01'
GROUP BY p.category, c.city_name
ORDER BY gmv DESC;
""")

任务：为电商平台设计订单分析的星型模型

【事实表】dwd_order_detail —— 订单明细事实表
----------------------------------------
度量字段（可以求和/平均的数值）：
  1. amount         DECIMAL —— 订单金额
  2. quantity        INT     —— 购买数量
  3. discount_amount DECIMAL —— 优惠金额
  4. shipping_fee    DECIMAL —— 运费
  5. pay_amount      DECIMAL —— 实付金额

维度外键（关联到维度表的ID）：
  1. user_id         BIGINT  → dim_user
  2. product_id      BIGINT  → dim_product
  3. shop_id         BIGINT  → dim_shop
  4. order_date      DATE    → dim_date
  5. city_id         INT     → dim_city
  6. channel_id      INT     → dim_channel

【维度表】
----------------------------------------
dim_user（用户维度）：
  user_id, name, phone, city, level, register_date, ...

dim_product（商品维度）：
  product_id, name, category, brand, price, status, ...

dim_shop（店铺维度）：
  shop_id, name, owner, city, open_date, level, ...

dim_date（时间维度）：
  date_id, year, month, day, weekday, is_holiday, week_of_year, quarter, ...

【查询示例】各品类 × 城市的GMV
----------------------------------------

SELECT
    p.categor

### 步骤2：进阶练习——DWD vs DWS性能对比实验

In [3]:
# ===== 步骤2：模拟DWD vs DWS的性能对比 =====

import time
import random

print("=" * 60)
print("实验：DWD vs DWS——同样的分析，差100倍的查询时间")
print("=" * 60)
print()

# 模拟：DWD层有5000万行订单明细
print("场景：计算'过去30天各品类的GMV和客单价'")
print()

print("方案A：查DWD层（订单明细表，5000万行）")
print("-" * 40)
print("  SELECT category,")
print("         SUM(amount) AS gmv,")
print("         SUM(amount)/COUNT(DISTINCT user_id) AS arpu")
print("  FROM dwd_order_detail")
print("  WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'")
print("  GROUP BY category;")
print()
print("  预估扫描：30天 × 约170万行/天 = 5000万行")
print("  预估耗时：30-60秒")
print()

print("方案B：查DWS层（品类日汇总表，300行）")
print("-" * 40)
print("  SELECT category,")
print("         SUM(daily_gmv) AS gmv,")
print("         SUM(daily_gmv)/SUM(daily_users) AS arpu")
print("  FROM dws_category_daily_1d")
print("  WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'")
print("  GROUP BY category;")
print()
print("  预估扫描：30天 × 10品类 = 300行")
print("  预估耗时：0.1-0.5秒")
print()

print("结论：DWS比DWD快100倍——代价是DWS不能按'品牌'维度拆（没有这个字段）")
print()
print("选择策略：")
print("  ✓ 维度在DWS中有 → 用DWS（快100倍）")
print("  ✓ 维度不在DWS中 → 用DWD（慢但灵活）")
print("  ✓ 经常用但DWS没有的维度 → 推动加到DWS层")

实验：DWD vs DWS——同样的分析，差100倍的查询时间

场景：计算'过去30天各品类的GMV和客单价'

方案A：查DWD层（订单明细表，5000万行）
----------------------------------------
  SELECT category,
         SUM(amount) AS gmv,
         SUM(amount)/COUNT(DISTINCT user_id) AS arpu
  FROM dwd_order_detail
  WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
  GROUP BY category;

  预估扫描：30天 × 约170万行/天 = 5000万行
  预估耗时：30-60秒

方案B：查DWS层（品类日汇总表，300行）
----------------------------------------
  SELECT category,
         SUM(daily_gmv) AS gmv,
         SUM(daily_gmv)/SUM(daily_users) AS arpu
  FROM dws_category_daily_1d
  WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
  GROUP BY category;

  预估扫描：30天 × 10品类 = 300行
  预估耗时：0.1-0.5秒

结论：DWS比DWD快100倍——代价是DWS不能按'品牌'维度拆（没有这个字段）

选择策略：
  ✓ 维度在DWS中有 → 用DWS（快100倍）
  ✓ 维度不在DWS中 → 用DWD（慢但灵活）
  ✓ 经常用但DWS没有的维度 → 推动加到DWS层


### 步骤3：挑战练习——从零设计数仓方案

In [4]:
# ===== 步骤3：为"内容社区"设计数仓分层方案 =====

print("=" * 60)
print("挑战：为一款内容社区App设计数仓分层方案")
print("=" * 60)
print()

print("业务背景：")
print("  - 用户可以发帖、评论、点赞、关注")
print("  - 业务库MySQL有三张核心表：")
print("    · posts（帖子表）")
print("    · comments（评论表）")
print("    · likes（点赞表）")
print()

print("你的设计方案：")
print()

print("【ODS层】")
print("  ods_posts_df      —— 照搬MySQL的posts表")
print("  ods_comments_df   —— 照搬MySQL的comments表")
print("  ods_likes_df      —— 照搬MySQL的likes表")
print()

print("【DWD层】")
print("  dwd_post_detail_df     —— 帖子明细（清洗+标准化+增加字段）")
print("    清洗内容：过滤敏感词、统一时间格式、处理NULL")
print("    增加字段：post_type（图文/视频/长文）、content_length")
print()
print("  dwd_comment_detail_df  —— 评论明细")
print("  dwd_like_detail_df     —— 点赞明细")
print()

print("【DWS层】")
print("  dws_user_daily_1d      —— 用户日报（发帖数、评论数、获赞数）")
print("  dws_post_daily_1d      —— 帖子日报（浏览量、评论数、点赞数）")
print("  dws_topic_daily_1d     —— 话题日报（帖子数、参与用户数）")
print()

print("【ADS层】")
print("  ads_creator_dashboard  —— 创作者看板（粉丝数、内容数、互动率）")
print("  ads_operation_daily    —— 运营日报（DAU、新增内容、互动率）")
print("  ads_trending_topics     —— 热门话题榜")
print()

print("【自检清单】")
checks = [
    "ODS是否保留了所有原始字段？",
    "DWD是否做了字段标准化？",
    "DWS是否覆盖了最常用的分析维度？",
    "ADS是否直接对应具体的业务应用？",
    "是否有循环依赖？",
    "命名是否规范（前缀+主题+粒度）？",
]
for i, check in enumerate(checks, 1):
    print(f"  [{i}] {check} ✓")

挑战：为一款内容社区App设计数仓分层方案

业务背景：
  - 用户可以发帖、评论、点赞、关注
  - 业务库MySQL有三张核心表：
    · posts（帖子表）
    · comments（评论表）
    · likes（点赞表）

你的设计方案：

【ODS层】
  ods_posts_df      —— 照搬MySQL的posts表
  ods_comments_df   —— 照搬MySQL的comments表
  ods_likes_df      —— 照搬MySQL的likes表

【DWD层】
  dwd_post_detail_df     —— 帖子明细（清洗+标准化+增加字段）
    清洗内容：过滤敏感词、统一时间格式、处理NULL
    增加字段：post_type（图文/视频/长文）、content_length

  dwd_comment_detail_df  —— 评论明细
  dwd_like_detail_df     —— 点赞明细

【DWS层】
  dws_user_daily_1d      —— 用户日报（发帖数、评论数、获赞数）
  dws_post_daily_1d      —— 帖子日报（浏览量、评论数、点赞数）
  dws_topic_daily_1d     —— 话题日报（帖子数、参与用户数）

【ADS层】
  ads_creator_dashboard  —— 创作者看板（粉丝数、内容数、互动率）
  ads_operation_daily    —— 运营日报（DAU、新增内容、互动率）
  ads_trending_topics     —— 热门话题榜

【自检清单】
  [1] ODS是否保留了所有原始字段？ ✓
  [2] DWD是否做了字段标准化？ ✓
  [3] DWS是否覆盖了最常用的分析维度？ ✓
  [4] ADS是否直接对应具体的业务应用？ ✓
  [5] 是否有循环依赖？ ✓
  [6] 命名是否规范（前缀+主题+粒度）？ ✓


## 📊 区块4：业务场景应用

### 场景1：DWD和DWS，选哪层

你要算"过去30天各品类的GMV和客单价"。

**选择A：查DWD层（订单明细表）**
```sql
SELECT category, SUM(amount) AS gmv,
       SUM(amount) / COUNT(DISTINCT user_id) AS arpu
FROM dwd_order_detail
WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
GROUP BY category;
```
- 优点：灵活，想要什么粒度都行
- 缺点：扫描5000万行，查询要跑30-60秒

**选择B：查DWS层（品类日汇总表）**
```sql
SELECT category, SUM(daily_gmv) AS gmv,
       SUM(daily_gmv) / SUM(daily_users) AS arpu
FROM dws_category_daily_1d
WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
GROUP BY category;
```
- 优点：只扫描300行，0.1秒出结果
- 缺点：如果突然想按"品牌"而不是"品类"看，DWS层没有品牌维度

**选择标准**：
1. 如果分析维度DWS已有 → 用DWS（快）
2. 如果需要DWS没有的维度 → 用DWD（灵活但慢）
3. 如果某个维度经常被用到但DWS没有 → 推动把这个维度加到DWS层

### 场景2：推动高频查询进入DWS层

你发现自己和团队每周至少5次要跑以下SQL：
```sql
SELECT city, DATE_FORMAT(order_date, '%Y-%m') AS month,
       SUM(amount) AS gmv
FROM dwd_order_detail
WHERE order_date >= '2025-01-01'
GROUP BY city, month;
```

每次跑30秒，5个分析师×52周×每次30秒 = 浪费130分钟/年。

**推动步骤**：
1. 在周会上提出："我建议把城市×月GMV的汇总加到DWS层，建一张`dws_city_monthly_1m`表。理由：全团队每周至少用5次，每次30秒，汇总后0.1秒。"
2. 和数仓同学对齐：需要哪些维度（城市、月份）、哪些指标（GMV、订单数、用户数）、数据粒度（月）
3. 提需求 → 数仓开发 → 验收数据 → 全团队切到新表

**回报**：以后这个查询变为0.1秒，全团队效率提升。

### 场景3：命名混乱导致的"找表地狱"

新同事入职，面对300多张表完全不知道哪个是哪个。表名毫无规律：
- `user_info_new` vs `user_info_v2` vs `user_info_final` —— 哪个是最新的？
- `order_stat` vs `order_stats` vs `order_statistics` —— 有什么区别？
- `tmp_202601_xxx` —— 能用吗？会被删吗？

你作为资深分析师，做了三件事：
1. 整理了"分析师常用表Top 20"文档，包括表名、层级、粒度、更新频率
2. 推动数仓统一命名规范：`{层级}_{主题}_{粒度}`
3. 跟新人说："不要用`tmp_`开头的表，不要用带`_final`、`_v2`的表——这些通常不是最终版本"

### 场景4：事实表和维度表的设计边界

产品经理提需求："我想在每个用户的订单里看到用户的最新等级。"

两种设计方案：

**方案A（不推荐）**：把user_level冗余到订单事实表
- 问题：用户等级会变——今天是"金牌"，明天可能是"银牌"。如果冗余过去了，历史订单的用户等级就错了

**方案B（推荐）**：订单事实表只存user_id，查询时JOIN用户维度表
- 如果需要"用户下单时的等级"（历史快照），用SCD Type 2处理
- 如果需要"用户最新等级"，直接用事实表JOIN维表即可

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出ODS/DWD/DWS/ADS各层的全称和作用
- [ ] 能根据表名前缀判断一张表属于哪一层
- [ ] 能区分事实表和维度表
- [ ] 能画出简单的星型模型（1张事实表+3-4张维度表）

**答案提示**：
- ODS: Operational Data Store，照搬原始数据；DWD: Data Warehouse Detail，清洗明细；DWS: Data Warehouse Summary，轻度汇总；ADS: Application Data Service，应用数据
- 看前缀：ods_/dwd_/dws_/ads_/dim_/tmp_
- 事实表记录事件（亿级行、追加），维度表记录实体属性（万级行、更新）
- 事实表在中间存度量和外键，维度表在外围存描述属性

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能根据分析需求选择查哪一层（DWD vs DWS）
- [ ] 能识别"该进DWS但还在DWD"的高频查询
- [ ] 能解释星型模型和雪花模型的选择依据
- [ ] 能画出部门级的数据流分层图

**答案提示**：
- 维度在DWS中有→用DWS；维度不在DWS中→用DWD；高频但DWS没有→推动加
- 高频查询（每周>5次、每次>10秒）应推动进DWS
- 分析型数仓优先星型模型（查询快、冗余可接受）
- 从ODS开始，画清楚每一层有哪些核心表

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能从分析师视角参与数仓设计讨论
- [ ] 能评估现有分层是否合理（哪层太厚、哪层太薄）
- [ ] 能在"查询性能"和"数据灵活性"之间做出合理权衡
- [ ] 能设计新业务线的完整数仓分层方案

**答案提示**：
- 评估维度：ODS是否够"原始"、DWD是否够"干净"、DWS是否覆盖高频维度、ADS是否直接对应应用
- 太厚=清洗和聚合做得不够，太多人重复清洗；太薄=过度分层，链路太长维护成本高
- 权衡原则：查询频率高→性能优先（DWS/ADS），灵活性要求高→DWD

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 昨日** | 数据流转链路 | 分层是数据流转的具体实现——ODS接收上游数据，ADS输出给下游应用 |
| **← 前置** | 第1-3周商业思维 | OSM/AARRR等框架需要数仓表来落地——你的指标定义最终体现在DWD/DWS的字段口径上 |
| **→ 明日** | SCD与宽表设计 | 维度表会变（SCD），宽表是分层的应用形态——今天理解了分层，明天才知道在哪个层处理变化 |
| **→ 后天** | Hive与大数据处理 | 数仓的分层表最终是靠Hive/Spark来存储和计算的 |
| **→ 第5-6周** | SQL分析实战 | 你写的SQL在查哪一层？这决定了查询性能和可用维度 |
| **业务连接** | 数据建设推动 | 分析师推动"高频查询进入DWS"是数据建设的重要贡献 |

### 横向连接

**与数据质量的关联**：
- DWD层是数据质量的"第一道防线"——清洗和标准化在这一层完成
- 如果DWD层的口径定义错了，下游所有层都错

**与指标体系的关联**：
- 指标的定义最终要落到具体的表和字段上
- DWS层是"原子指标"和"派生指标"的最佳存储位置

**与数据治理的关联**：
- 分层是数据治理的基础设施
- 没有分层就谈不上数据血缘、数据质量、数据安全

## 📝 区块7：当日小结

> **数仓分四层不是"为了复杂而复杂"——每一层都有不可替代的作用。ODS照搬数据（可追溯），DWD清洗标准化（可信赖），DWS预聚合（查询快），ADS定制化（直接好用）。最经典的建模方式是星型模型——事实表在中间记录"发生了什么"，维度表在外围描述"谁/什么/何时/何地"。数据分析师不是数仓的"用户"，而是"需求方"和"共建者"。当你发现自己反复在DWD层做同样的聚合时，就应该推动这个聚合进入DWS层——这是你对数仓的贡献。**

**三条金句带回家**：
1. ODS不能跳过——它是数仓的"定海神针"，是可追溯性的唯一保证
2. DWS层的价值不在"能算"，而在"已经算好了等你"
3. 分析师推动一个汇总进入DWS层，等于给全团队每天省下N个30秒

### 🔖 课后拓展

1. **练习**：找出你每天使用的3张核心表，判断它们属于哪一层。如果命名不规范，尝试用ODS/DWD/DWS/ADS的前缀重新命名它们

2. **思考**：如果你需要设计一张"用户行为宽表"（包含用户基本信息+最近30天行为指标+历史累计指标），你会放在哪一层？为什么？

3. **预习**：明天要学的SCD（缓慢变化维度）解决的是"历史变了但历史不能变"的矛盾——今天你理解了维度表，明天你会理解维度表的变化

4. **面试准备**：背诵以下面试题答案：
   - "数仓为什么分四层？"
   - "事实表和维度表的区别？"
   - "星型模型和雪花模型怎么选？"
   - "你怎么决定一个指标该放DWD还是DWS？"

---

*03analyst_thinking · Week 4 Day 2 · 数仓建模一*